In [1]:
import pandas as pd
import numpy as np
from utils.data import *

GLOBAL_START='2024-01-01'
GLOBAL_END='2026-03-01'

In [2]:

price_chain_df, hsi_df, vhsi_df, future = get_data()
price_chain_df_clean, hsi_df_clean, vhsi_df_clean, index_df = prepare_datasets(
    price_chain_df, hsi_df, vhsi_df, future
)
merged_df=pd.merge(price_chain_df_clean, index_df, on="Date", how="left")
index_features=add_index_features(merged_df)
smirk_daily = fit_iv_surface_shape_daily(
    df=price_chain_df_clean,                  # 你的 option chain dataframe
    price_col="SettlementPrice",   # 用来算 parity 的价格列
    iv_col="ImpliedVolatility",
    rate_col=None,           # 没有无风险利率就先 None
    r_default=0.0,
    dte_col="dte",
    x_band=np.inf,
    min_points=5,
)

option_features=add_option_shape_daily_features(
    shape_df=smirk_daily)

agg_features = build_periodic_features(
    index_features=index_features,
    option_features=option_features,
    lookback=3,   # 1M history
    pred_len=1,   # predict next 1M regime
)



e:\Project\6000C_AIOptionTrading\utils\data.py:109: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  hsi[col] = hsi[col].fillna(method="ffill")


In [6]:
from utils.context import build_news_context_from_db
news_context = await build_news_context_from_db(
    start_date="2024-08-01",
    end_date="2024-09-01",
    keyword=None,
    limit=100,
    fulltext_chars=300,
)

TypeError: unsupported operand type(s) for |: 'type' and 'NoneType'

In [ ]:
from utils.llm import call_llm, choose_scenario

reasoning, res=call_llm(as_of_date="2024-09-01")
scenario=choose_scenario(res)


In [ ]:
from utils.backtest import compute_backtest_metrics

metrics=compute_backtest_metrics(scenario, entry_date="2024-08-01", exit_date="2024-09-01", option_df=merged_df)


In [ ]:
from db.operations import save_performance_row
save_performance_row(metrics)